# Comparison report

Look at the same movies across scorers, dimensions, and detectors, then read the Knight-style tables for the detector you select.

Edit only the configuration cell. Re-run the notebook after a change. Pinned movies stay fixed. Random movies stay fixed until you change `RANDOM_SEED`.

Window width is already stored in `model_scores`. This notebook reads those windows. A different window width means re-running scoring, then `extract-reversals`.

Detector settings below are applied in memory. They do not rewrite `reversal_features`. The summary, Pearson, and regression tables use the rows already stored for `DETECTOR_NAME`. Live stats and trajectory plots use the detector objects built in the configuration cell, so you can try a parameter before extracting again.

Run `plot-shift extract-reversals` first so the stored tables are not empty.

In [ ]:
%matplotlib inline

from plot_shift.reversals import create_reversal_detector

DB_PATH = "../data/proceeded/data.db"

# Stored features and the detector used for the plots below.
DETECTOR_NAME = "ruptures"
TREND_WINDOW_SIZE = 3
RUPTURES_PENALTY = 1.0

# Set SECOND_DETECTOR_NAME to None to skip the side-by-side check.
SECOND_DETECTOR_NAME = "trendet"
SECOND_TREND_WINDOW_SIZE = 5
SECOND_RUPTURES_PENALTY = 1.0

RANDOM_SEED = 7
N_RANDOM = 4
PINNED_IMDB_IDS: list[int] = []

# None keeps every scorer or dimension.
SCORER_NAME: str | None = None
DIMENSION: str | None = None

detector = create_reversal_detector(
    DETECTOR_NAME,
    window_size=TREND_WINDOW_SIZE,
    ruptures_penalty=RUPTURES_PENALTY,
)
second_detector = None
if SECOND_DETECTOR_NAME:
    second_detector = create_reversal_detector(
        SECOND_DETECTOR_NAME,
        window_size=SECOND_TREND_WINDOW_SIZE,
        ruptures_penalty=SECOND_RUPTURES_PENALTY,
    )

## Coverage and chosen movies

`n_windows` is the stored window count. `window_duration` is seconds per window and varies by film, because each movie is split into the same number of windows. The Knight tables in the next section use only `DETECTOR_NAME`.

In [ ]:
from plot_shift.data.db import PlotShiftSqliteDB
from plot_shift.evaluation import comparison
from plot_shift.evaluation import reversal_report as rr

db = PlotShiftSqliteDB(DB_PATH).__enter__()
features = rr.load_features(db)
if SCORER_NAME is not None:
    features = features[features["scorer_name"] == SCORER_NAME]
if DIMENSION is not None:
    features = features[features["dimension"] == DIMENSION]

selected = features[features["detector_name"] == DETECTOR_NAME].copy()
movie_ids = comparison.pick_movies(selected, N_RANDOM, RANDOM_SEED, PINNED_IMDB_IDS)

missing = [imdb_id for imdb_id in movie_ids if imdb_id not in set(selected["imdb_id_int"])]
if missing:
    print("Pinned ids with no stored features for this detector:", missing)

comparison.coverage(features)

In [ ]:
order = {imdb_id: index for index, imdb_id in enumerate(movie_ids)}
chosen = selected.loc[
    selected["imdb_id_int"].isin(movie_ids),
    ["imdb_id_int", "primary_title", "start_year", "average_rating"],
].drop_duplicates()
chosen = (
    chosen.assign(_order=chosen["imdb_id_int"].map(order))
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)
chosen

## Knight tables

One group per scorer and dimension, for `DETECTOR_NAME` only. Predictors in the regression are standardized inside each group, matching Knight's Table 1: rating on reversal count and mean intensity.

In [ ]:
rr.summary_stats(selected)

In [ ]:
rr.pearson_table(selected)

In [ ]:
rr.regression_table(selected)

## Trajectories

Each panel is one stored window series. Purple is an upward trend, blue is downward, and red points are the reversal extrema. These plots use the in-memory detector, not the stored feature row.

In [ ]:
figures = comparison.plot_cases(
    db,
    movie_ids,
    detector,
    scorer_name=SCORER_NAME,
    dimension=DIMENSION,
)

### Second detector

Same movies, other detector. Skipped when `SECOND_DETECTOR_NAME` is `None`.

In [ ]:
if second_detector is None:
    print("No second detector.")
else:
    figures = comparison.plot_cases(
        db,
        movie_ids,
        second_detector,
        scorer_name=SCORER_NAME,
        dimension=DIMENSION,
    )

## Live reversal stats

Computed now from the stored windows. Compare `detector_name` on a pinned movie after you change `TREND_WINDOW_SIZE` or `RUPTURES_PENALTY` and re-run.

In [ ]:
import pandas as pd

live = comparison.live_stats(
    db,
    movie_ids,
    detector,
    scorer_name=SCORER_NAME,
    dimension=DIMENSION,
)
if second_detector is not None:
    live = pd.concat(
        [
            live,
            comparison.live_stats(
                db,
                movie_ids,
                second_detector,
                scorer_name=SCORER_NAME,
                dimension=DIMENSION,
            ),
        ],
        ignore_index=True,
    )
live

## Series agreement

Pearson *r* between the windowed series of each scorer/dimension pair, on the movies above. This is the internal comparison: do two scorers rise and fall in the same windows? It does not use the detector. `mean_r` ignores pairs whose window counts differ (`aligned` is false).

In [ ]:
detail, summary = comparison.series_agreement(
    db,
    movie_ids,
    scorer_name=SCORER_NAME,
    dimension=DIMENSION,
)
summary

In [ ]:
detail